# PTB-XL ECG Machine Learning Capstone: Clustering Track
## Objective: Unsupervised Discovery of ECG Phenotype Groups

This notebook implements the **Clustering Track** under the Capstone rubric:

1. **Data Preparation**: Same feature matrix as Classification tracks, full dataset (unsupervised).
   Guard-gated median imputation and z-score standardisation applied.
2. **Dimensionality Reduction**: PCA to 2 components for all 2-D visualisations.
3. **K-Means Clustering**
   - Elbow method (WCSS) over $k \in \{2, 3, 4, 5, 6, 7, 8, 9, 10\}$
   - Deterministic silhouette scores to confirm optimal $k$ (BUG-10 fixed)
   - Cluster characterisation: mean feature profiles + dominant diagnostic label per cluster
4. **Agglomerative Hierarchical Clustering**
   - Ward linkage with dendrogram (truncated to top 30 merges)
   - Connectivity graph ($k$-NN) to avoid $O(N^2)$ memory bottleneck (BUG-11 fixed)
   - Applied at optimal $k$ from K-Means; cluster profiles compared
5. **External Validity**: Cluster purity w.r.t. dominant diagnostic labels, confusion heatmap.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Helvetica', 'Arial', 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
palette = sns.color_palette('tab10')

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Clustering libraries loaded.")


---
## Section A: Data Ingestion & Preprocessing


In [ ]:
# Path resolution (BUG-17)
cwd = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(cwd, '..')) if os.path.basename(cwd) == 'notebooks' else os.path.abspath(cwd)
DATA_DIR = os.path.join(BASE_DIR, 'data')

features_df = pd.read_csv(os.path.join(DATA_DIR, 'stage4_features.csv'))
labels_df   = pd.read_csv(os.path.join(DATA_DIR, 'ptbxl_labels.csv'))

label_targets = ['label_NORM', 'label_MI', 'label_STTC', 'label_CD', 'label_HYP']
df = features_df.merge(
    labels_df[['ecg_id', 'fold'] + label_targets],
    on='ecg_id', how='inner'
)

# Build dominant label for external validity
priority_cols = ['label_MI', 'label_CD', 'label_HYP', 'label_STTC', 'label_NORM']
class_names   = ['MI', 'CD', 'HYP', 'STTC', 'NORM']

def collapse_dominant_label(row):
    for col, name in zip(priority_cols, class_names):
        if row[col] == 1:
            return name
    return 'UNKNOWN'

df['dominant_label'] = df.apply(collapse_dominant_label, axis=1)

print(f"Records: {len(df)} | Labels: {df['dominant_label'].value_counts().to_dict()}")


In [ ]:
# Feature matrix discovery (BUG-03: exclude delineation_ok)
exclude_cols = {
    'ecg_id', 'patient_id', 'fold', 'dominant_label', 'delineation_ok',
    'label_NORM', 'label_MI', 'label_STTC', 'label_CD', 'label_HYP'
}
feature_cols = [c for c in df.columns if c not in exclude_cols
                and pd.api.types.is_numeric_dtype(df[c])]

# Engineer qtc_qrs_ratio using correct column names (BUG-02)
if 'qtc_mean_ms' in df.columns and 'qrs_dur_mean_ms' in df.columns:
    df['qtc_qrs_ratio'] = df['qtc_mean_ms'] / (df['qrs_dur_mean_ms'].replace(0, np.nan))
    if 'qtc_qrs_ratio' not in feature_cols:
        feature_cols.append('qtc_qrs_ratio')

# Apply guard masking on unverified conduction intervals (BUG-04)
guard_cols = [
    'pr_mean_ms', 'pr_std_ms', 'qrs_dur_mean_ms', 'qrs_dur_std_ms',
    'qt_mean_ms', 'qt_std_ms', 'qtc_mean_ms'
]
df.loc[~df['delineation_ok'].astype(bool), guard_cols] = np.nan

print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

X_raw = df[feature_cols].values

imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()
X = scaler.fit_transform(imputer.fit_transform(X_raw))

print(f"Final feature matrix shape: {X.shape}")


---
## Section B: PCA Dimensionality Reduction for Visualisation


In [ ]:
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca2.fit_transform(X)

pca_full = PCA(random_state=RANDOM_STATE).fit(X)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n90 = int(np.searchsorted(cumvar, 0.90)) + 1

print(f"PC1 explains {pca2.explained_variance_ratio_[0]*100:.1f}%")
print(f"PC2 explains {pca2.explained_variance_ratio_[1]*100:.1f}%")
print(f"Components needed for 90% variance: {n90}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_ * 100, color=palette[0], edgecolor='white')
axes[0].set_xlabel('Principal Component', fontsize=11)
axes[0].set_ylabel('Explained Variance (%)', fontsize=11)
axes[0].set_title('Scree Plot', fontsize=13, fontweight='bold')

axes[1].plot(range(1, len(cumvar) + 1), cumvar * 100, marker='o', color=palette[1], linewidth=2)
axes[1].axhline(90, color='red', linestyle='--', label='90% threshold')
axes[1].axvline(n90, color='grey', linestyle=':', label=f'{n90} components')
axes[1].set_xlabel('Number of Components', fontsize=11)
axes[1].set_ylabel('Cumulative Explained Variance (%)', fontsize=11)
axes[1].set_title('Cumulative Variance Explained', fontsize=13, fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()


---
## Section C: K-Means Clustering

### C.1 — Elbow Method & Silhouette Analysis (BUG-10 Fixed Deterministic Subsample)


In [ ]:
K_RANGE = range(2, 11)
wcss        = []
silhouettes = []

# Deterministic subsample for silhouette evaluation across all k values (BUG-10)
rng = np.random.RandomState(RANDOM_STATE)
eval_idx = rng.choice(X.shape[0], min(5000, X.shape[0]), replace=False)

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=20,
                max_iter=300, random_state=RANDOM_STATE)
    km.fit(X)
    wcss.append(km.inertia_)
    sil = silhouette_score(X[eval_idx], km.labels_[eval_idx])
    silhouettes.append(sil)
    print(f"  k={k:2d}  WCSS={km.inertia_:,.0f}  Silhouette={sil:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(K_RANGE), wcss, marker='o', color=palette[0], linewidth=2)
axes[0].set_xticks(list(K_RANGE))
axes[0].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[0].set_ylabel('WCSS (Inertia)', fontsize=11)
axes[0].set_title('Elbow Method — WCSS', fontsize=13, fontweight='bold')

axes[1].plot(list(K_RANGE), silhouettes, marker='s', color=palette[1], linewidth=2)
axes[1].set_xticks(list(K_RANGE))
axes[1].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_title('Silhouette Score vs k', fontsize=13, fontweight='bold')

plt.suptitle('K-Means: Optimal k Selection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

OPTIMAL_K = int(np.argmax(silhouettes)) + 2
print(f"Optimal k by silhouette: {OPTIMAL_K}")


### C.2 — Final K-Means at Optimal k


In [ ]:
km_final = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=50,
                  max_iter=500, random_state=RANDOM_STATE)
km_labels = km_final.fit_predict(X)
df['km_cluster'] = km_labels

print(f"K-Means (k={OPTIMAL_K}) cluster sizes:")
print(pd.Series(km_labels).value_counts().sort_index())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cm_palette = sns.color_palette('tab10', OPTIMAL_K)
for c in range(OPTIMAL_K):
    mask = km_labels == c
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    s=4, alpha=0.5, color=cm_palette[c], label=f'C{c}')
axes[0].set_xlabel('PC1', fontsize=11)
axes[0].set_ylabel('PC2', fontsize=11)
axes[0].set_title(f'K-Means (k={OPTIMAL_K}) — PCA 2-D Projection', fontsize=12, fontweight='bold')
axes[0].legend(markerscale=3, fontsize=9)

label_palette = {l: c for l, c in zip(class_names, sns.color_palette('Set2', 5))}
for lbl in class_names:
    mask = df['dominant_label'] == lbl
    axes[1].scatter(X_2d[mask.values, 0], X_2d[mask.values, 1],
                    s=4, alpha=0.4, color=label_palette[lbl], label=lbl)
axes[1].set_xlabel('PC1', fontsize=11)
axes[1].set_ylabel('PC2', fontsize=11)
axes[1].set_title('True Diagnostic Labels — PCA 2-D Projection', fontsize=12, fontweight='bold')
axes[1].legend(markerscale=3, fontsize=9)

plt.suptitle('Cluster vs. Label Structure in 2-D PCA Space', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
profile_df = df[feature_cols + ['km_cluster']].groupby('km_cluster').mean()
print("K-Means Cluster Feature Profiles (mean per cluster):")
display(profile_df.T.round(3))


In [ ]:
cluster_label_counts = (df.groupby(['km_cluster', 'dominant_label'])
                          .size()
                          .unstack(fill_value=0))
cluster_label_counts['Total']     = cluster_label_counts.sum(axis=1)
cluster_label_counts['Majority']  = cluster_label_counts[class_names].max(axis=1)
cluster_label_counts['Purity']    = (cluster_label_counts['Majority'] /
                                     cluster_label_counts['Total']).round(3)
cluster_label_counts['DomLabel']  = cluster_label_counts[class_names].idxmax(axis=1)

print("K-Means Cluster Purity w.r.t. Diagnostic Labels:")
display(cluster_label_counts)

overall_purity = (cluster_label_counts['Majority'].sum() /
                  cluster_label_counts['Total'].sum())
print(f"Overall K-Means Purity: {overall_purity:.4f}")


In [ ]:
plt.figure(figsize=(9, max(4, OPTIMAL_K)))
sns.heatmap(cluster_label_counts[class_names], annot=True, fmt='d',
            cmap='YlOrRd', linewidths=0.5)
plt.title(f'K-Means (k={OPTIMAL_K}): Cluster × Diagnostic Label Count',
          fontsize=13, fontweight='bold')
plt.xlabel('Diagnostic Category', fontsize=11)
plt.ylabel('Cluster ID', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
TSNE_SAMPLE = min(5000, X.shape[0])
np.random.seed(RANDOM_STATE)
tsne_idx = np.random.choice(X.shape[0], TSNE_SAMPLE, replace=False)

print(f"Computing t-SNE projection on subsample (n={TSNE_SAMPLE})...")
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
X_tsne = tsne.fit_transform(X[tsne_idx])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

tsne_km_labels = km_labels[tsne_idx]
for c in range(OPTIMAL_K):
    mask = tsne_km_labels == c
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    s=6, alpha=0.5, color=cm_palette[c], label=f'C{c}')
axes[0].set_xlabel('t-SNE Dimension 1', fontsize=11)
axes[0].set_ylabel('t-SNE Dimension 2', fontsize=11)
axes[0].set_title(f'K-Means (k={OPTIMAL_K}) — t-SNE 2-D Manifold', fontsize=12, fontweight='bold')
axes[0].legend(markerscale=3, fontsize=9)

tsne_true_labels = df['dominant_label'].iloc[tsne_idx].values
for lbl in class_names:
    mask = tsne_true_labels == lbl
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    s=6, alpha=0.4, color=label_palette[lbl], label=lbl)
axes[1].set_xlabel('t-SNE Dimension 1', fontsize=11)
axes[1].set_ylabel('t-SNE Dimension 2', fontsize=11)
axes[1].set_title('True Diagnostic Labels — t-SNE 2-D Manifold', fontsize=12, fontweight='bold')
axes[1].legend(markerscale=3, fontsize=9)

plt.suptitle('t-SNE Non-Linear Manifold Visualisation (Subsample n=5,000)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## Section D: Agglomerative Hierarchical Clustering (Ward Linkage - BUG-11 Memory Fix)


In [ ]:
DENDRO_SAMPLE = min(3000, X.shape[0])
np.random.seed(RANDOM_STATE)
didx = np.random.choice(X.shape[0], DENDRO_SAMPLE, replace=False)
Z = linkage(X[didx], method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90,
           show_contracted=True, ax=ax, color_threshold=0.7 * max(Z[:, 2]))
ax.set_title('Agglomerative Hierarchical Clustering — Ward Linkage Dendrogram', fontsize=12, fontweight='bold')
ax.set_xlabel('Sample Index / Cluster Size', fontsize=11)
ax.set_ylabel('Ward Distance', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Agglomerative clustering with k-NN connectivity graph to prevent O(N^2) memory crash (BUG-11)
print("Constructing k-nearest neighbors connectivity graph...")
knn_graph = kneighbors_graph(X, n_neighbors=15, include_self=False)

agg = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage='ward', connectivity=knn_graph)
agg_labels = agg.fit_predict(X)
df['agg_cluster'] = agg_labels

print(f"Agglomerative (k={OPTIMAL_K}) cluster sizes:")
print(pd.Series(agg_labels).value_counts().sort_index())


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
agg_palette = sns.color_palette('tab10', OPTIMAL_K)
for c in range(OPTIMAL_K):
    mask = agg_labels == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               s=4, alpha=0.5, color=agg_palette[c], label=f'C{c}')
ax.set_xlabel('PC1', fontsize=11)
ax.set_ylabel('PC2', fontsize=11)
ax.set_title(f'Agglomerative (Ward, k={OPTIMAL_K}) — PCA 2-D Projection',
             fontsize=12, fontweight='bold')
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
agg_label_counts = (df.groupby(['agg_cluster', 'dominant_label'])
                      .size()
                      .unstack(fill_value=0))
agg_label_counts['Total']    = agg_label_counts.sum(axis=1)
agg_label_counts['Majority'] = agg_label_counts[class_names].max(axis=1)
agg_label_counts['Purity']   = (agg_label_counts['Majority'] /
                                 agg_label_counts['Total']).round(3)
agg_label_counts['DomLabel'] = agg_label_counts[class_names].idxmax(axis=1)

print("Agglomerative Cluster Purity w.r.t. Diagnostic Labels:")
display(agg_label_counts)

agg_purity = (agg_label_counts['Majority'].sum() / agg_label_counts['Total'].sum())
print(f"Overall Agglomerative Purity: {agg_purity:.4f}")


In [ ]:
plt.figure(figsize=(9, max(4, OPTIMAL_K)))
sns.heatmap(agg_label_counts[class_names], annot=True, fmt='d',
            cmap='YlGnBu', linewidths=0.5)
plt.title(f'Agglomerative (Ward, k={OPTIMAL_K}): Cluster × Diagnostic Label Count',
          fontsize=13, fontweight='bold')
plt.xlabel('Diagnostic Category', fontsize=11)
plt.ylabel('Cluster ID', fontsize=11)
plt.tight_layout()
plt.show()


---
## Section E: Summary Comparison — K-Means vs Agglomerative


In [ ]:
km_sil  = silhouette_score(X, km_labels, sample_size=min(5000, len(X)), random_state=RANDOM_STATE)
agg_sil = silhouette_score(X, agg_labels, sample_size=min(5000, len(X)), random_state=RANDOM_STATE)

km_db  = davies_bouldin_score(X, km_labels)
agg_db = davies_bouldin_score(X, agg_labels)

km_ch  = calinski_harabasz_score(X, km_labels)
agg_ch = calinski_harabasz_score(X, agg_labels)

summary = pd.DataFrame({
    'Method':              ['K-Means', 'Agglomerative (Ward)'],
    'k':                   [OPTIMAL_K, OPTIMAL_K],
    'Silhouette Score':    [round(km_sil, 4),   round(agg_sil, 4)],
    'Davies-Bouldin':      [round(km_db, 4),    round(agg_db, 4)],
    'Calinski-Harabasz':   [round(km_ch, 2),    round(agg_ch, 2)],
    'Overall Purity':      [round(overall_purity, 4), round(agg_purity, 4)],
})

print("=" * 75)
print("CLUSTERING ALGORITHM COMPARISON")
print("=" * 75)
display(summary)
